In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("HardcodedData") \
    .getOrCreate()


schema = StructType([
    StructField("id",        IntegerType(), False),  # False = null nahi ho sakta
    StructField("name",      StringType(),  True),
    StructField("age",      IntegerType(), True),
    StructField("city",    StringType(),  True),
    StructField("salary",    DoubleType(),  True)
])

# Data banao
data = [
    (1, "Rahul",  25, "Delhi",   45000.0),
    (2, "Priya",  30, "Mumbai",  62000.0),
    (3, "Amit",   22, "Pune",    38000.0),
    (4, "Sneha",  28, "Chennai", 55000.0),
    (5, "Vikram", 35, "Delhi",   78000.0)
]

# DataFrame banao
df = spark.createDataFrame(data, schema)

# Data dikhao
df.show()

# Schema dikhao
df.printSchema()




+---+------+---+-------+-------+
| id|  name|age|   city| salary|
+---+------+---+-------+-------+
|  1| Rahul| 25|  Delhi|45000.0|
|  2| Priya| 30| Mumbai|62000.0|
|  3|  Amit| 22|   Pune|38000.0|
|  4| Sneha| 28|Chennai|55000.0|
|  5|Vikram| 35|  Delhi|78000.0|
+---+------+---+-------+-------+

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)



In [3]:
test_data = "/data/datasets/Tests_data/customer.csv"

df_2 = spark.read.format('csv').option('header','true').option('inferSchema','true').load(test_data)

# df_2  = spark.read.csv("datasets/Tests_data/customer.csv")

df_2.printSchema()

df_2.show(5)


root
 |-- customer_id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_zip: integer (nullable = true)

+-----------+-------------------+------------------+------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|
+-----------+-------------------+------------------+------------+
|          1|               Jane|            Connor|       22801|
|          2|             Manuel|              Diaz|       22821|
|          3|                Bob|            Wilson|       22821|
|          4|             Deanna|        Washington|       22801|
|          5|            Abigail|            Harris|       22801|
+-----------+-------------------+------------------+------------+
only showing top 5 rows



In [4]:
# Number of partitions after reading data
print(f"Number of partitions : {df_2.rdd.getNumPartitions()}")

Number of partitions : 1


In [8]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

'134217728b'

In [19]:
# Select columns in different options
from pyspark.sql.functions import *

# df_2.select('customer_first_name').show(10)
df_2.select('customer_first_name','customer_zip').show(2)
df_2.select(col('customer_first_name'),col("customer_zip")).show(4)
df_2.select(col("customer_zip").alias("cust_zip") ,col("customer_first_name").alias("cust_First_Name")).show(5)



+-------------------+------------+
|customer_first_name|customer_zip|
+-------------------+------------+
|               Jane|       22801|
|             Manuel|       22821|
+-------------------+------------+
only showing top 2 rows

+-------------------+------------+
|customer_first_name|customer_zip|
+-------------------+------------+
|               Jane|       22801|
|             Manuel|       22821|
|                Bob|       22821|
|             Deanna|       22801|
+-------------------+------------+
only showing top 4 rows

+--------+---------------+
|cust_zip|cust_First_Name|
+--------+---------------+
|   22801|           Jane|
|   22821|         Manuel|
|   22821|            Bob|
|   22801|         Deanna|
|   22801|        Abigail|
+--------+---------------+
only showing top 5 rows



In [30]:
# Drive new column using withColumn

df_3 = df_2.withColumn("full_name", concat(col("customer_first_name"), lit(" "), col("customer_last_name")))
df_3.select("full_name").show(5)

+-----------------+
|        full_name|
+-----------------+
|      Jane Connor|
|      Manuel Diaz|
|       Bob Wilson|
|Deanna Washington|
|   Abigail Harris|
+-----------------+
only showing top 5 rows



In [39]:
# Filter Conditions

df_3.filter(col("full_name") == "Jane Connor").show()

df_3.filter(col("full_name").isin("Manuel Diaz","Bob Wilson")).show()

df_3.filter(col("full_name").contains("ul")).show()

df_3.filter(col("full_name").like("%a")).show()

df_3.filter(col("full_name").like("a%")).show()

+-----------+-------------------+------------------+------------+-----------+
|customer_id|customer_first_name|customer_last_name|customer_zip|  full_name|
+-----------+-------------------+------------------+------------+-----------+
|          1|               Jane|            Connor|       22801|Jane Connor|
+-----------+-------------------+------------------+------------+-----------+

+-----------+-------------------+------------------+------------+-----------+
|customer_id|customer_first_name|customer_last_name|customer_zip|  full_name|
+-----------+-------------------+------------------+------------+-----------+
|          2|             Manuel|              Diaz|       22821|Manuel Diaz|
|          3|                Bob|            Wilson|       22821| Bob Wilson|
+-----------+-------------------+------------------+------------+-----------+

+-----------+-------------------+------------------+------------+---------------+
|customer_id|customer_first_name|customer_last_name|custom

In [43]:
# Get distinct values

df_3.distinct().show(5)
df_3.dropDuplicates().show(5)

+-----------+-------------------+------------------+------------+----------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|       full_name|
+-----------+-------------------+------------------+------------+----------------+
|          8|              Norma|        Valenzuela|       22821|Norma Valenzuela|
|         23|              Alvin|            Laurie|       22801|    Alvin Laurie|
|         22|             George|               Rai|       22801|      George Rai|
|          3|                Bob|            Wilson|       22821|      Bob Wilson|
|         15|            Darrell|           Messina|       22801| Darrell Messina|
+-----------+-------------------+------------------+------------+----------------+
only showing top 5 rows

+-----------+-------------------+------------------+------------+----------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|       full_name|
+-----------+-------------------+------------------+----------

In [52]:
# Arrange data using order by

df_3.orderBy(col("customer_id").desc()).show(5)

df_3.distinct().orderBy(col("customer_zip").asc(),col("full_name").desc()).show(5)

+-----------+-------------------+------------------+------------+---------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|      full_name|
+-----------+-------------------+------------------+------------+---------------+
|         26|             Tracie|          Goehring|       22821|Tracie Goehring|
|         25|             Bonnie|            Hassan|       22801|  Bonnie Hassan|
|         24|               Dawn|              Nale|       22801|      Dawn Nale|
|         23|              Alvin|            Laurie|       22801|   Alvin Laurie|
|         22|             George|               Rai|       22801|     George Rai|
+-----------+-------------------+------------------+------------+---------------+
only showing top 5 rows

+-----------+-------------------+------------------+------------+---------------+
|customer_id|customer_first_name|customer_last_name|customer_zip|      full_name|
+-----------+-------------------+------------------+------------+--------

In [53]:
# Group By

df_3.groupBy("customer_zip").agg(count("*").alias("total_customer")).show(5)

+------------+--------------+
|customer_zip|total_customer|
+------------+--------------+
|       22802|             4|
|       22801|            15|
|       22821|             7|
+------------+--------------+



In [56]:
accum = spark.sparkContext.accumulator(0)


df_3.foreach(lambda row: accum.add(row["customer_id"]))

print(accum.value)

351


In [61]:
# Case statement in

df_3.withColumn("Location" , when(col("customer_zip") == 22802 , "urban_area")\
                .when(col("customer_zip") == 22821, "semi_urban")\
                .otherwise("village")).show(8)

+-----------+-------------------+------------------+------------+-----------------+----------+
|customer_id|customer_first_name|customer_last_name|customer_zip|        full_name|  Location|
+-----------+-------------------+------------------+------------+-----------------+----------+
|          1|               Jane|            Connor|       22801|      Jane Connor|   village|
|          2|             Manuel|              Diaz|       22821|      Manuel Diaz|semi_urban|
|          3|                Bob|            Wilson|       22821|       Bob Wilson|semi_urban|
|          4|             Deanna|        Washington|       22801|Deanna Washington|   village|
|          5|            Abigail|            Harris|       22801|   Abigail Harris|   village|
|          6|              Betty|           Bullard|       22801|    Betty Bullard|   village|
|          7|            Jessica|           Armenta|       22821|  Jessica Armenta|semi_urban|
|          8|              Norma|        Valenzuel

In [63]:
# Window function

from pyspark.sql.window import Window

window_sep = Window.partitionBy("customer_zip").orderBy(col("full_name").asc())
df_3.withColumn("dense_rank", dense_rank().over(window_sep)).show(5)

+-----------+-------------------+------------------+------------+--------------+----------+
|customer_id|customer_first_name|customer_last_name|customer_zip|     full_name|dense_rank|
+-----------+-------------------+------------------+------------+--------------+----------+
|          5|            Abigail|            Harris|       22801|Abigail Harris|         1|
|         16|                Ada|            Nieves|       22801|    Ada Nieves|         2|
|         23|              Alvin|            Laurie|       22801|  Alvin Laurie|         3|
|          6|              Betty|           Bullard|       22801| Betty Bullard|         4|
|         25|             Bonnie|            Hassan|       22801| Bonnie Hassan|         5|
+-----------+-------------------+------------------+------------+--------------+----------+
only showing top 5 rows



In [76]:
from pyspark.sql.window import Window

window_sep = Window.partitionBy("customer_zip").orderBy(col("customer_last_name").asc())
result = df_3.withColumn("dense_rank", dense_rank().over(window_sep))

In [77]:
result.write.format('csv').option('header','true').option('delimiter',',').save('outputs/first_pyspark/result/')

In [78]:
print(result)

DataFrame[customer_id: int, customer_first_name: string, customer_last_name: string, customer_zip: int, full_name: string, dense_rank: int]
